In [1]:
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
suppressMessages(library('Biobase'))
suppressMessages(library(gplots))
suppressMessages(library(BiocParallel))
suppressMessages(library(cowplot))
suppressMessages(library(GenomeInfoDb))
suppressMessages(library("stringr"))
suppressMessages(library(tidyverse))

Warning message:
“package ‘hdf5r’ was built under R version 4.2.2”


# Create pseudobulk matrices

## functions

In [12]:
#function which takes in a list of long format atac_fragment dfs with 
#sample names (df), an overall windows file (windows) and then makes these 
#into sparse matrices and merges them together

merge_sparse_matrices_hvws <- function(dfs, windows){
    samples <- names(dfs)
    for (sample in samples){
        #get missing windows list for this sample
        print(paste(sample,Sys.time(),sep=': '))
        df <- dfs[[sample]]
        mis_windows <- windows[!windows %in% levels(df$V1)]
        
        #make sure there are missing_windows
        if (length(mis_windows) > 0){
            print('Adding missing windows')
            #create a new long format matrix (sm) with the missing windows added as 0 counts
            filler_bc <- as.character(df$V2[[1]])
            print(paste("Using filler BC:",filler_bc,sep=" "))
            new_rows <- cbind(as.data.frame(mis_windows),
                              as.data.frame(rep(filler_bc),length(mis_windows)),
                              as.data.frame(rep(0,length(mis_windows))))
            colnames(new_rows) <- c("V1","V2","V3")
            lfm <- rbind(df,new_rows)
        #if there aren't, set lfm to df
        } else {
            print('No windows were missing')
            lfm <- df
        }
        
        #cut down lfm to just be the hvws (windows)
        lfm_cut <- lfm[lfm$V1 %in% windows,]
        
        #set the levels of the lfm based on the desired bc order and reorder V1
        lfm_cut$V1 <- factor(lfm_cut$V1, levels=windows)
        lfm2 <- lfm_cut[order(lfm_cut$V1),]
        lfm2$V2 <- as.factor(lfm2$V2)
        
        if (sample == samples[1]){
            #if first sample, will make the overall sparse matrix 
            overall_sm <- with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(overall_sm))
            
        } else {
            #otherwise, convert into a sparse matrix and add to the overall one
            sm = with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(sm))
            overall_sm = cbind(overall_sm, sm) 
        }
    }
    return(overall_sm)
}

## Major cell types

In [6]:
#Read in LFMs
cells<-scan('/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/celltypes.txt', what="", sep="\n")
lfms<-list()
for (cell in cells){
    lfms[[cell]]<-read.table(sprintf('/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/%s/%s.long_fmt_mtx.txt.gz', cell, cell), sep="\t", header=FALSE, stringsAsFactors=FALSE)
    
}

In [7]:
#Reformat lfms
fixed_lfms<-lapply(1:16, function(i){
    name<-names(lfms)[i]
    df<-lfms[[i]]
    df<-relocate(df, V2)
    colnames(df)<-c('V1', 'V2', 'V3')
    return(df)
})

In [2]:
names(fixed_lfms)<-cells


ERROR: Error in eval(expr, envir, enclos): object 'cells' not found


In [11]:
peaks<-scan('/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/unified_peaks_output/nPODUnionPeaks_reformatted.tsv', sep="\n", what="")
peaks2 <- sort(peaks)
head(peaks2)


[1] "1:1000020-1000320"     "1:100028799-100029099" "1:100037929-100038229"
[4] "1:100038647-100038947" "1:100046900-100047200" "1:100051493-100051692"

In [13]:
overall_sm<-merge_sparse_matrices_hvws(fixed_lfms, peaks2)

[1] "Acinar_1_2_6: 2024-05-27 20:46:55"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACAGCCAGTAAAGC"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049 128325
[1] "Acinar_3: 2024-05-27 20:49:35"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACATGCACCCACAG"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049  24340
[1] "Acinar_5: 2024-05-27 20:50:15"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACCGGCAGCAAATA"


...

[1] 414049     13
[1] "Quiescent_Stellate: 2024-05-27 20:51:20"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_ACAACAACATTGTGAT"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049    976
[1] "LymphEndo: 2024-05-27 20:51:22"
[1] "Adding missing windows"
[1] "Using filler BC: MM_339_CCTGGGAGTATTTGCG"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049     65


In [15]:
### Function to sum up all counts for a cell type by sample
### 7/12/23: adapted to also work for multiple cell types (aka an all_but situation)

get_per_sample_gex_SUMS <- function(sparsematrix, cell_type, samples, outdir, all_but=FALSE){
    if (all_but == TRUE){
        bcs <- meta[!meta$FinalSubtypes == cell_type,]$bc
        prefix <- 'all_but_'
    } else {
        bcs <- meta[meta$FinalSubtypes == cell_type,]$bc
        prefix <- ''
    }
    
    #pull out rows of gex.counts where BC Ident matches cell_type
    counts <- sparsematrix[,colnames(sparsematrix) %in% bcs]

    #go through samples and calculate sum of gex values
    gex_list <- list()
    for (sample in samples){
        sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
        counts.cut <- counts[,sample_cols]
        
        #if only one bc, this becomes a vector which is an issue
        if (typeof(counts.cut) == 'double'){
            tot.counts <- round(counts.cut)
        #if there are NO bcs, this will return NA (just return 0 for everything)
        } else if(length(colnames(counts.cut)) == 0){
            tot.counts <- rep(0,length(row.names(counts)))
        } else {
            tot.counts <- round(rowSums(counts.cut))
        }
        #add total counts to list
        gex_list[[sample]] <- tot.counts            
    }
    
    fin.counts.df <- as.data.frame(gex_list)
    # print(head(fin.counts.df))
    colnames(fin.counts.df) <- as.character(colnames(fin.counts.df))
    #export df
    mtx.fp <- file.path(outdir,sprintf('%s%s_sample_gex_total_counts.txt',prefix,cell_type))
    write.table(fin.counts.df,mtx.fp,sep='\t',quote=FALSE)
}

### create matrices

In [16]:
outdir<-'pseudobulk_mtx/'
dir.create(outdir)

In [19]:
atac <- readRDS('/nfs/lab/projects/nPOD/clustering_files/ATAC/20230518_snATAC_finalandFixedPeaksIncl_correctedFixPeak_nPODids_noMultiomeMM426.rds')
atac

An object of class Seurat 
1084915 features across 174598 samples within 5 assays 
Active assay: Peaks_011823_final (368688 features, 0 variable features)
 4 other assays present: ATAC, ACTIVITY, ACTIVITY_allFeatures, FixPeaks_011823
 3 dimensional reductions calculated: lsi, harmony.atac, umap.atac

In [20]:
#Read in meta data
meta<- atac@meta.data 

,orig.ident,nCount_ATAC,nFeature_ATAC,library,technology,sex,condition,ATAC_snn_res.0.5,seurat_clusters,nCount_RNA,⋯,prediction.score.Mast,FinalSubtypes,AcinarSumPrediction,condition_subtype,nCount_Peaks_011823_final,nFeature_Peaks_011823_final,nCount_FixPeaks_011823,nFeature_FixPeaks_011823,library2,nPOD_ID
,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<fct>,<fct>,<int>,⋯,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<int>,<chr>,<chr>
MM_339_AAACGAATCCCGAAGC-1,MM,748,715,MM_339,snatac,M,T1D,1,1,8258,⋯,0,Acinar_1_2_6,1,earlyT1D,836,813,716,699,MM_339,6247
MM_339_AAACGAATCTATGAGC-1,MM,830,802,MM_339,snatac,M,T1D,2,2,8182,⋯,0,Acinar_1_2_6,1,earlyT1D,939,893,819,789,MM_339,6247
MM_339_AAACTCGGTGTGCTTA-1,MM,505,489,MM_339,snatac,M,T1D,1,1,5412,⋯,0,Acinar_1_2_6,1,earlyT1D,575,560,486,476,MM_339,6247
MM_339_AAAGATGTCGGATGTT-1,MM,746,726,MM_339,snatac,M,T1D,1,1,7378,⋯,0,Acinar_1_2_6,1,earlyT1D,773,748,649,634,MM_339,6247
MM_339_AAAGGATAGGAGTCTG-1,MM,621,602,MM_339,snatac,M,T1D,1,1,5897,⋯,0,Acinar_1_2_6,1,earlyT1D,644,628,554,540,MM_339,6247
MM_339_AAAGGATAGTCGAGCA-1,MM,834,797,MM_339,snatac,M,T1D,1,1,8898,⋯,0,Acinar_1_2_6,1,earlyT1D,913,882,777,762,MM_339,6247


In [21]:
meta$bc <- substr(rownames(meta),1,nchar(rownames(meta))-2)
#Create a list of barcodes for each sample
samples<-as.character(unique(meta$nPOD_ID))
sample_bcs<-list()
for (samp in samples){
    bars<-meta[meta$nPOD_ID == samp,]
    barcodes <- bars$bc
    sample_bcs[[samp]]<-barcodes
}
head(sample_bcs[[1]])

[1] "MM_339_AAACGAATCCCGAAGC" "MM_339_AAACGAATCTATGAGC"
[3] "MM_339_AAACTCGGTGTGCTTA" "MM_339_AAAGATGTCGGATGTT"
[5] "MM_339_AAAGGATAGGAGTCTG" "MM_339_AAAGGATAGTCGAGCA"

In [22]:
#create mtx for all bcs for the subpop of interest
for (cell in cells){
    get_per_sample_gex_SUMS(overall_sm, cell, samples, outdir, all_but=FALSE)
}

In [23]:
#create mtx for all bcs EXCEPT the subpop of interest (for comparison)
for (cell in cells){
    get_per_sample_gex_SUMS(overall_sm, cell, samples, outdir, all_but=TRUE)
}

### Making normalized counts matrices

In [24]:
#Collecting info on peak sizes (can just pic one file since all have same peaks)
df<-read.table('pseudobulk_mtx/Beta_sample_gex_total_counts.txt', sep="\t", header=TRUE)
peak_list<-rownames(df)
peak_list2<-gsub(":", "-", peak_list)
pls<-str_split_fixed(string = peak_list2, pattern = "-", n = 3)
pls<-as.data.frame(pls)
#string to integer
pls[,2]<-strtoi(pls[,2])
pls[,3]<-strtoi(pls[,3])
final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))


,peak_id,chrom,start,end,peak.sizes
,<chr>,<chr>,<int>,<int>,<int>
1,1:1000020-1000320,1,1000020,1000320,300
2,1:100028799-100029099,1,100028799,100029099,300
3,1:100037929-100038229,1,100037929,100038229,300
4,1:100038647-100038947,1,100038647,100038947,300
5,1:100046900-100047200,1,100046900,100047200,300
6,1:100051493-100051692,1,100051493,100051692,199


In [25]:
write.table(x = final_peaklist, file = 'pseudobulk_mtx/Peak_sizes.txt', quote = FALSE, sep="\t")


## Now collecting info on per sample sequencing depth

In [26]:
samples <- unique(atac$nPOD_ID)
celltypes <- unique(atac$FinalSubtypes)

[1] "6247" "6418" "6339" "6197" "6380" "6375" "6282" "6264" "6431" "6479"
[11] "6450" "6362" "6220" "6310" "6229" "6456" "6405" "6267" "6228" "6401"
[21] "6234" "6236" "6424" "6520" "6429" "6521" "6393" "6512" "6505"

[1] "Acinar_1_2_6"       "Acinar_5"           "Acinar_3"          
 [4] "Acinar_4"           "Ductal"             "Activated_Stellate"
 [7] "Endothelial"        "Schwann"            "Alpha"             
[10] "Delta"              "Beta"               "Macrophage"        
[13] "Tcells"             "Mast"               "Quiescent_Stellate"
[16] "LymphEndo"          "Bcells"             "MUC5b_Ductal"

In [31]:
for(cell in celltypes){
    samp_depth<-matrix(nrow=length(samples), ncol=2)
    colnames(samp_depth)<-c('Sample', 'Depth')
    samp_depth[,1]<-samples
    samp_depth<-as.data.frame(samp_depth)
    # first get depth for all celltypes individually 
    ct_atac <- subset(atac, subset=FinalSubtypes==cell)
    for (samp in samples){
        if(nrow(ct_atac@meta.data[ct_atac$nPOD_ID == samp,]) == 0) next
        df<-subset(ct_atac, subset=nPOD_ID==samp)
        r_sum<-rowSums(df[["Peaks_011823_final"]]@counts)
        c_sum<-colSums(df[["Peaks_011823_final"]]@counts)
        all_sum<-sum(r_sum,c_sum)
        samp_depth$Depth[samp_depth$Sample == samp]<-all_sum
    }
    out_file <- paste0('pseudobulk_mtx/',cell,'_Samp_depth.txt')
    write.table(samp_depth, out_file, sep='\t', quote=FALSE)
}



In [32]:
for(cell in celltypes){
    samp_depth<-matrix(nrow=length(samples), ncol=2)
    colnames(samp_depth)<-c('Sample', 'Depth')
    samp_depth[,1]<-samples
    samp_depth<-as.data.frame(samp_depth)
    # second get depth for everything but celltype 
    ct_atac <- subset(atac, subset=FinalSubtypes!=cell)
    for (samp in samples){
        if(nrow(ct_atac@meta.data[ct_atac$nPOD_ID == samp,]) == 0) next
        df<-subset(ct_atac, subset=nPOD_ID==samp)
        r_sum<-rowSums(df[["Peaks_011823_final"]]@counts)
        c_sum<-colSums(df[["Peaks_011823_final"]]@counts)
        all_sum<-sum(r_sum,c_sum)
        samp_depth$Depth[samp_depth$Sample == samp]<-all_sum
    }
    out_file <- paste0('pseudobulk_mtx/allbut_',cell,'_Samp_depth.txt')
    write.table(samp_depth, out_file,sep='\t', quote=FALSE)
}



In [33]:
cells<-scan('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/celltypes.txt', what="", sep="\n")
cells

[1] "Acinar_1_2_6"       "Acinar_3"           "Acinar_5"          
 [4] "Acinar_4"           "Ductal"             "Activated_Stellate"
 [7] "Endothelial"        "Schwann"            "Alpha"             
[10] "Delta"              "Beta"               "Macrophage"        
[13] "Tcells"             "Mast"               "Quiescent_Stellate"
[16] "LymphEndo"

In [35]:
#Loop for all pseudobulked matricies
final_peaklist<-read.table('pseudobulk_mtx/Peak_sizes.txt', sep="\t", header=TRUE)
#samp_depth<-read.table('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/pseudobulk_mtx/Samp_depth.txt', sep="\t", header=TRUE)

for (cell in cells){
    df<-read.table(sprintf('pseudobulk_mtx/%s_sample_gex_total_counts.txt', cell), sep="\t", header=TRUE, check.names = FALSE)
    colnames(df)<- sub("^X", "", names(df))
    df = df[,colSums(df) > 0]
    norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
    colnames(norm_counts)<-colnames(df)
    rownames(norm_counts)<-rownames(df)
    #### read in sample depth for THIS CELL TYPE
    ct_file <-  paste0('pseudobulk_mtx/',cell,'_Samp_depth.txt')
    samp_depth<-read.table(ct_file, sep="\t", header=TRUE)
    samp_depth<- na.omit(samp_depth)
    
    for (i in 1:ncol(df)){
        raw.counts<-as.numeric(df[,i])
        sample<-colnames(df)[i]
        depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
        w_size<-as.numeric(final_peaklist$peak.sizes)
        norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
        norm_counts[,i]<-norm_count
        if (i == 1){
            message(paste0("Starting ", cell, " at ", Sys.time()))
            }
        }
    write.table(norm_counts, sprintf('pseudobulk_mtx/%s_sample_gex_CMP_CORRECTED2.txt', cell), sep="\t", quote=FALSE)
    
    
    #### everything but celltype
    df2<-read.table(sprintf('pseudobulk_mtx/all_but_%s_sample_gex_total_counts.txt', cell), sep="\t", header=TRUE, check.names = FALSE)
    colnames(df2)<- sub("^X", "", names(df2))
    df2 = df2[,colSums(df2) > 0]

    norm_counts<-matrix(nrow=nrow(df2), ncol=ncol(df2))
    colnames(norm_counts)<-colnames(df2)
    rownames(norm_counts)<-rownames(df2)
     #### read in sample depth for EVERYTHING BUT CELL TYPE
    ct_file <-  paste0('pseudobulk_mtx/allbut_',cell,'_Samp_depth.txt')
    samp_depth<-read.table(ct_file, sep="\t", header=TRUE)
    samp_depth<- na.omit(samp_depth)

    for (i in 1:ncol(df2)){
        raw.counts<-as.numeric(df2[,i])
        sample<-colnames(df2)[i]
        depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
        w_size<-as.numeric(final_peaklist$peak.sizes)
        #Next line is normalizing for read depth, peak size, scaling factor AND number of celltypes since this table has a bunch of cell types summed together
        norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
        norm_counts[,i]<-norm_count
        if (i == 37){
            message(paste0("Finished at ", Sys.time()))
        }
    }
    write.table(norm_counts, sprintf('pseudobulk_mtx/all_but_%s_sample_gex_CMP_CORRECTED2.txt', cell), sep="\t", quote=FALSE)
}

Starting Acinar_1_2_6 at 2024-05-27 23:48:23

Starting Acinar_3 at 2024-05-27 23:48:44

Starting Acinar_5 at 2024-05-27 23:49:05

Starting Acinar_4 at 2024-05-27 23:49:25

Starting Ductal at 2024-05-27 23:49:46

Starting Activated_Stellate at 2024-05-27 23:50:08

Starting Endothelial at 2024-05-27 23:50:29

Starting Schwann at 2024-05-27 23:50:50

Starting Alpha at 2024-05-27 23:51:06

Starting Delta at 2024-05-27 23:51:27

Starting Beta at 2024-05-27 23:51:47

Starting Macrophage at 2024-05-27 23:52:08

Starting Tcells at 2024-05-27 23:52:29

Starting Mast at 2024-05-27 23:52:50

Starting Quiescent_Stellate at 2024-05-27 23:53:06

Starting LymphEndo at 2024-05-27 23:53:26



## Acinar subtypes -- to compare between one acinar subtype to all other acinar cells

In [38]:
#Read in LFMs
cells<-c('Acinar_1_2_6','Acinar_5','Acinar_3','Acinar_4')#scan('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/celltypes.txt', what="", sep="\n")
lfms<-list()
for (cell in cells){
    lfms[[cell]]<-read.table(sprintf('%s/%s.long_fmt_mtx.txt.gz', cell, cell), sep="\t", header=FALSE, stringsAsFactors=FALSE)
}

In [39]:
#Reformat lfms
fixed_lfms<-lapply(1:4, function(i){
    name<-names(lfms)[i]
    df<-lfms[[i]]
    df<-relocate(df, V2)
    colnames(df)<-c('V1', 'V2', 'V3')
    return(df)
})
names(fixed_lfms)<-cells
## read in peaks
peaks<-scan('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/unified_peaks_output/nPODUnionPeaks_reformatted.tsv', sep="\n", what="")
peaks2 <- sort(peaks)


,V1,V2,V3
,<chr>,<chr>,<int>
1,10:100009314-100010390,6229_sort_AAACAGCCAGTAAAGC,6
2,10:100747941-100748150,6229_sort_AAACAGCCAGTAAAGC,1
3,10:100826673-100828495,6229_sort_AAACAGCCAGTAAAGC,2
4,10:100912428-100913537,6229_sort_AAACAGCCAGTAAAGC,4
5,10:100996927-100998658,6229_sort_AAACAGCCAGTAAAGC,6
6,10:100999115-100999971,6229_sort_AAACAGCCAGTAAAGC,2


[1] "1:1000020-1000320"     "1:100028799-100029099" "1:100037929-100038229"
[4] "1:100038647-100038947" "1:100046900-100047200" "1:100051493-100051692"

In [40]:
overall_sm <-merge_sparse_matrices_hvws(fixed_lfms, peaks2)

[1] "Acinar_1_2_6: 2024-05-28 06:29:01"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACAGCCAGTAAAGC"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049 128325
[1] "Acinar_5: 2024-05-28 06:32:01"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACCGGCAGCAAATA"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049   1862
[1] "Acinar_3: 2024-05-28 06:32:04"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AAACATGCACCCACAG"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049  24340
[1] "Acinar_4: 2024-05-28 06:32:39"
[1] "Adding missing windows"
[1] "Using filler BC: 6229_sort_AGGTGAGGTCCGCTGT"


Warning message in as.data.frame.vector(x, ..., nm = nm):
"'row.names' is not a character vector of length 1 -- omitting it. Will be an error!"


[1] 414049    552


In [41]:
#Create a list of barcodes for each cell type
cell_bars<-list()
for (cell in cells){
    df<-fixed_lfms[[cell]]
    bars<-unique(df$V2)
    cell_bars[[cell]]<-bars
}
head(cell_bars[[1]])

[1] "6229_sort_AAACAGCCAGTAAAGC" "6229_sort_AAACATGCATACCCGG"
[3] "6229_sort_AAACATGCATTGCAGC" "6229_sort_AAACCAACAATAATCC"
[5] "6229_sort_AAACCGAAGTAAGAAC" "6229_sort_AAACCGCGTCCGGTTC"

In [43]:
#Read in meta data
meta<-atac@meta.data #read.table('/nfs/lab/welison/multiome/chromvar/230522_ChromVar_Outputs/atac.metadata.tsv', sep="\t", header=TRUE)
head(meta)

meta$bc <- substr(rownames(meta),1,nchar(rownames(meta))-2)

#Create a list of barcodes for each sample
samples<-as.character(unique(meta$nPOD_ID))
sample_bcs<-list()
for (samp in samples){
    bars<-meta[meta$nPOD_ID == samp,]
    barcodes <- bars$bc
    sample_bcs[[samp]]<-barcodes
}
head(sample_bcs[[1]])

#colnames(counts) <- as.character(colnames(counts))

,orig.ident,nCount_ATAC,nFeature_ATAC,library,technology,sex,condition,ATAC_snn_res.0.5,seurat_clusters,nCount_RNA,⋯,prediction.score.Mast,FinalSubtypes,AcinarSumPrediction,condition_subtype,nCount_Peaks_011823_final,nFeature_Peaks_011823_final,nCount_FixPeaks_011823,nFeature_FixPeaks_011823,library2,nPOD_ID
,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<fct>,<fct>,<int>,⋯,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<int>,<chr>,<chr>
MM_339_AAACGAATCCCGAAGC-1,MM,748,715,MM_339,snatac,M,T1D,1,1,8258,⋯,0,Acinar_1_2_6,1,earlyT1D,836,813,716,699,MM_339,6247
MM_339_AAACGAATCTATGAGC-1,MM,830,802,MM_339,snatac,M,T1D,2,2,8182,⋯,0,Acinar_1_2_6,1,earlyT1D,939,893,819,789,MM_339,6247
MM_339_AAACTCGGTGTGCTTA-1,MM,505,489,MM_339,snatac,M,T1D,1,1,5412,⋯,0,Acinar_1_2_6,1,earlyT1D,575,560,486,476,MM_339,6247
MM_339_AAAGATGTCGGATGTT-1,MM,746,726,MM_339,snatac,M,T1D,1,1,7378,⋯,0,Acinar_1_2_6,1,earlyT1D,773,748,649,634,MM_339,6247
MM_339_AAAGGATAGGAGTCTG-1,MM,621,602,MM_339,snatac,M,T1D,1,1,5897,⋯,0,Acinar_1_2_6,1,earlyT1D,644,628,554,540,MM_339,6247
MM_339_AAAGGATAGTCGAGCA-1,MM,834,797,MM_339,snatac,M,T1D,1,1,8898,⋯,0,Acinar_1_2_6,1,earlyT1D,913,882,777,762,MM_339,6247


[1] "MM_339_AAACGAATCCCGAAGC" "MM_339_AAACGAATCTATGAGC"
[3] "MM_339_AAACTCGGTGTGCTTA" "MM_339_AAAGATGTCGGATGTT"
[5] "MM_339_AAAGGATAGGAGTCTG" "MM_339_AAAGGATAGTCGAGCA"

In [44]:
outdir <- 'pseudobulk_mtx/acinar_subtype/'
dir.create(outdir)

In [45]:
#create mtx for all bcs for the subpop of interest
for (cell in cells){
    get_per_sample_gex_SUMS(overall_sm, cell, samples, outdir, all_but=FALSE)
}

#create mtx for all bcs EXCEPT the subpop of interest (for comparison)
for (cell in cells){
    get_per_sample_gex_SUMS(overall_sm, cell, samples, outdir, all_but=TRUE)
}

In [46]:
#Collecting info on peak sizes (can just pic one file since all have same peaks)
df<-read.table('pseudobulk_mtx/acinar_subtype/Acinar_1_2_6_sample_gex_total_counts.txt', sep="\t", header=TRUE)
peak_list<-rownames(df)
peak_list2<-gsub(":", "-", peak_list)
pls<-str_split_fixed(string = peak_list2, pattern = "-", n = 3)
pls<-as.data.frame(pls)
#string to integer
pls[,2]<-strtoi(pls[,2])
pls[,3]<-strtoi(pls[,3])
final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))
head(final_peaklist)

write.table(x = final_peaklist, file = 'pseudobulk_mtx/acinar_subtype/Peak_sizes.txt', quote = FALSE, sep="\t")


,peak_id,chrom,start,end,peak.sizes
,<chr>,<chr>,<int>,<int>,<int>
1,1:1000020-1000320,1,1000020,1000320,300
2,1:100028799-100029099,1,100028799,100029099,300
3,1:100037929-100038229,1,100037929,100038229,300
4,1:100038647-100038947,1,100038647,100038947,300
5,1:100046900-100047200,1,100046900,100047200,300
6,1:100051493-100051692,1,100051493,100051692,199


In [47]:
#atac <- readRDS('/nfs/lab/projects/nPOD/clustering_files/ATAC/20230518_snATAC_finalandFixedPeaksIncl_correctedFixPeak_nPODids_noMultiomeMM426.rds')
samples <- unique(atac$nPOD_ID)
celltypes <- c('Acinar_1_2_6','Acinar_5','Acinar_3','Acinar_4')


An object of class Seurat 
1084915 features across 174598 samples within 5 assays 
Active assay: Peaks_011823_final (368688 features, 0 variable features)
 4 other assays present: ATAC, ACTIVITY, ACTIVITY_allFeatures, FixPeaks_011823
 3 dimensional reductions calculated: lsi, harmony.atac, umap.atac

[1] "6247" "6418" "6339" "6197" "6380" "6375" "6282" "6264" "6431" "6479"
[11] "6450" "6362" "6220" "6310" "6229" "6456" "6405" "6267" "6228" "6401"
[21] "6234" "6236" "6424" "6520" "6429" "6521" "6393" "6512" "6505"

[1] "Acinar_1_2_6" "Acinar_5"     "Acinar_3"     "Acinar_4"

In [48]:
for(cell in celltypes){
    samp_depth<-matrix(nrow=length(samples), ncol=2)
    colnames(samp_depth)<-c('Sample', 'Depth')
    samp_depth[,1]<-samples
    samp_depth<-as.data.frame(samp_depth)
    # first get depth for all celltypes individually 
    ct_atac <- subset(atac, subset=FinalSubtypes==cell)
    for (samp in samples){
        if(nrow(ct_atac@meta.data[ct_atac$nPOD_ID == samp,]) == 0) next
        df<-subset(ct_atac, subset=nPOD_ID==samp)
        r_sum<-rowSums(df[["Peaks_011823_final"]]@counts)
        c_sum<-colSums(df[["Peaks_011823_final"]]@counts)
        all_sum<-sum(r_sum,c_sum)
        samp_depth$Depth[samp_depth$Sample == samp]<-all_sum
    }
    out_file <- paste0('pseudobulk_mtx/acinar_subtype/',cell,'_Samp_depth.txt')
    write.table(samp_depth, out_file, sep='\t', quote=FALSE)
}



In [49]:
for(cell in celltypes){
    samp_depth<-matrix(nrow=length(samples), ncol=2)
    colnames(samp_depth)<-c('Sample', 'Depth')
    samp_depth[,1]<-samples
    samp_depth<-as.data.frame(samp_depth)
    # second get depth for everything but celltype 
    ct_atac <- subset(atac, subset=FinalSubtypes!=cell)
    for (samp in samples){
        if(nrow(ct_atac@meta.data[ct_atac$nPOD_ID == samp,]) == 0) next
        df<-subset(ct_atac, subset=nPOD_ID==samp)
        r_sum<-rowSums(df[["Peaks_011823_final"]]@counts)
        c_sum<-colSums(df[["Peaks_011823_final"]]@counts)
        all_sum<-sum(r_sum,c_sum)
        samp_depth$Depth[samp_depth$Sample == samp]<-all_sum
    }
    out_file <- paste0('pseudobulk_mtx/acinar_subtype/allbut_',cell,'_Samp_depth.txt')
    write.table(samp_depth, out_file,sep='\t', quote=FALSE)
}



In [51]:
#Loop for all pseudobulked matricies
final_peaklist<-read.table('pseudobulk_mtx/acinar_subtype/Peak_sizes.txt', sep="\t", header=TRUE)
#samp_depth<-read.table('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/pseudobulk_mtx/Samp_depth.txt', sep="\t", header=TRUE)

for (cell in cells){
    df<-read.table(sprintf('pseudobulk_mtx/acinar_subtype/%s_sample_gex_total_counts.txt', cell), sep="\t", header=TRUE, check.names = FALSE)
    colnames(df)<- sub("^X", "", names(df))
    df = df[,colSums(df) > 0]
    norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
    colnames(norm_counts)<-colnames(df)
    rownames(norm_counts)<-rownames(df)
    #### read in sample depth for THIS CELL TYPE
    ct_file <-  paste0('pseudobulk_mtx/acinar_subtype/',cell,'_Samp_depth.txt')
    samp_depth<-read.table(ct_file, sep="\t", header=TRUE)
    samp_depth<- na.omit(samp_depth)
    
    for (i in 1:ncol(df)){
        raw.counts<-as.numeric(df[,i])
        sample<-colnames(df)[i]
        depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
        w_size<-as.numeric(final_peaklist$peak.sizes)
        norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
        norm_counts[,i]<-norm_count
        if (i == 1){
            message(paste0("Starting ", cell, " at ", Sys.time()))
            }
        }
    write.table(norm_counts, sprintf('pseudobulk_mtx/acinar_subtype/%s_sample_gex_CMP_CORRECTED2.txt', cell), sep="\t", quote=FALSE)
    
    
    #### everything but celltype
    df2<-read.table(sprintf('pseudobulk_mtx/acinar_subtype/all_but_%s_sample_gex_total_counts.txt', cell), sep="\t", header=TRUE, check.names = FALSE)
    colnames(df2)<- sub("^X", "", names(df2))
    df2 = df2[,colSums(df2) > 0]

    norm_counts<-matrix(nrow=nrow(df2), ncol=ncol(df2))
    colnames(norm_counts)<-colnames(df2)
    rownames(norm_counts)<-rownames(df2)
     #### read in sample depth for EVERYTHING BUT CELL TYPE
    ct_file <-  paste0('pseudobulk_mtx/acinar_subtype/allbut_',cell,'_Samp_depth.txt')
    samp_depth<-read.table(ct_file, sep="\t", header=TRUE)
    samp_depth<- na.omit(samp_depth)

    for (i in 1:ncol(df2)){
        raw.counts<-as.numeric(df2[,i])
        sample<-colnames(df2)[i]
        depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
        w_size<-as.numeric(final_peaklist$peak.sizes)
        #Next line is normalizing for read depth, peak size, scaling factor AND number of celltypes since this table has a bunch of cell types summed together
        norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
        norm_counts[,i]<-norm_count
        if (i == 37){
            message(paste0("Finished at ", Sys.time()))
        }
    }
    write.table(norm_counts, sprintf('pseudobulk_mtx/acinar_subtype/all_but_%s_sample_gex_CMP_CORRECTED2.txt', cell), sep="\t", quote=FALSE)
}

Starting Acinar_1_2_6 at 2024-05-28 09:30:16

Starting Acinar_5 at 2024-05-28 09:30:50

Starting Acinar_3 at 2024-05-28 09:31:20

Starting Acinar_4 at 2024-05-28 09:31:52



# Logistic regression

# Fxn and loop for all cells

In [14]:
#Get your list of cell types
cells<-scan('/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/celltypes.txt', what="", sep="\n")
cells

[1] "Acinar_1_2_6"       "Acinar_3"           "Acinar_5"          
 [4] "Acinar_4"           "Ductal"             "Activated_Stellate"
 [7] "Endothelial"        "Schwann"            "Alpha"             
[10] "Delta"              "Beta"               "Macrophage"        
[13] "Tcells"             "Mast"               "Quiescent_Stellate"
[16] "LymphEndo"

In [16]:
#Read in meta data you want to use
metadata<-read.table('/nfs/lab/projects/nPOD/nPOD_metadata.txt', sep="\t", header=TRUE)
metadata$nPOD_ID <- as.character(metadata$Sample.ID)
metadata <- metadata[colnames(metadata) %in% c('nPOD_ID','Condition_subtype')]

In [18]:
#Define input and output directories
indir<-'pseudobulk_mtx/'
outdir<-'052824_correctedResults_allNDSamples_wAAb/'
getwd()
#dir.create(outdir) 

[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724"

In [19]:
#Function that basically transposes the data and merges it with metadata so
#that glm() can be used on a single dataframe.

#NOTE!! You will need to factorize the predicted variable and any covariates, in this case the Celltype and Sample
#(Donor), respectively.
get_input_df<-function(celltype, df_interest, df_other, meta_data){
    dfa<-as.data.frame(t(df_interest))
    dfo<-as.data.frame(t(df_other))
    dfa$nPOD_ID<-rownames(dfa)
    dfo$nPOD_ID<-rownames(dfo)
    dfa$Celltype<-celltype
    dfo$Celltype<-paste0("Not",celltype)
    dfall<-rbind(dfa,dfo)
    int<-left_join(x = dfall, y=metadata, by = 'nPOD_ID',multiple = "all")#,relationship = "many-to-many")
    int$Celltype<-as.factor(int$Celltype)
    int$nPOD_ID<-as.factor(int$nPOD_ID)
    final<-int
}

In [25]:
##Running logistic regression
meta <- read.table('/nfs/lab/projects/nPOD/nPOD_metadata.txt', sep='\t', header = TRUE)
all_disease <- meta[meta$Condition_subtype %in% c('T1D_early','T1D_late'),]$Sample.ID
samples <- unique(meta$Sample.ID)
nd <- setdiff(samples, all_disease)

for (cell in cells){
    message(paste0("Starting ",cell," at ", Sys.time()))
    #Read in inputs where "interest" is counts matrix for a given cell type and "other" is counts matrix for all
    #other cell types combined
    interest<-read.table(paste0(indir,cell,'_sample_gex_CMP_CORRECTED2.txt'), sep="\t", header=TRUE, check.names = FALSE)
    colnames(interest) <- gsub('X','',colnames(interest))
    nd_interest <- interest#[,colnames(interest)%in% nd]
    other<-read.table(paste0(indir,'all_but_',cell,'_sample_gex_CMP_CORRECTED2.txt'), sep="\t", header=TRUE,check.names = FALSE)
    colnames(other) <- gsub('X','',colnames(other))
    nd_other <- other#[,colnames(other)%in% nd]
    #Reads in files and gets input formatted
    input<-get_input_df(cell, nd_interest, nd_other, metadata)
        message(paste0("Starting log reg ",cell," at ", Sys.time()))

    #Runs logistic regression in parallel on 20 cores (can change number of cores with mc.cores)
    res_list<-mclapply(1:(ncol(input)-3), function(i) {
        res<-tidy(glm(formula=Celltype ~ input[ ,i] + nPOD_ID, family="binomial", data=input))
        pvalue<-res[2,5]
        ster<-res[2,3]
        peak<-colnames(input)[i]
        c(peak,pvalue, ster)
    }, mc.cores=60, mc.preschedule = TRUE)
    #Combines results list into dataframe
    res1<-matrix(unlist(res_list), ncol=3, byrow= TRUE)
    res1<-as.data.frame(res1)
    colnames(res1)<-c("peak","pvalue", "std.error")
    
    #Calculates log2foldchange from normalized counts matricies you read in and adds this into results dataframe
    L2FC_list<-mclapply(1:nrow(res1), function(i) {
        peak<-res1$peak[i]
        mean_interest<-mean(as.numeric(interest[peak,]))
        mean_other<-mean(as.numeric(other[peak, ]))
        FC<-(mean_interest/mean_other)
        L2FC<-log2(FC)
        c(peak,L2FC)
    }, mc.cores=60,mc.preschedule = TRUE)
    addinL2FC<-matrix(unlist(L2FC_list), ncol=2, byrow=TRUE)
    addinL2FC<-as.data.frame(addinL2FC)
    colnames(addinL2FC)<-c('peak', 'L2FC')
    rownames(addinL2FC)<-addinL2FC$peak
    newdf<-left_join(res1, addinL2FC)
    newdf$pvalue<-as.numeric(newdf$pvalue)
    #Adds in bonferroni corrected padj values as a column
    numpeaks<-nrow(newdf)
    newdf$padj<-p.adjust(p = newdf$pvalue, method = "BH", n = nrow(newdf))
    sorteddf<-arrange(newdf, pvalue, desc(L2FC))
    #Writes the final table to output directory
    write.table(sorteddf, paste0(outdir,cell,'.LogRegOut.txt'),  sep="\t", quote=FALSE, row.names=FALSE)
    
    #Prints the number of significantly enriched peaks for each celltype based on padj ≤ 0.1
    numsig<-nrow(sorteddf[sorteddf$padj <= 0.1, ])
    message(paste("There were ", numsig, " significantly enriched peaks for ", cell, " cells relative to other celltypes."))
    message("Finished ", cell, " ", Sys.time())
}


Starting Acinar_1_2_6 at 2024-05-29 12:33:37

Starting log reg Acinar_1_2_6 at 2024-05-29 12:33:59

Joining with `by = join_by(peak)`
There were  382866  significantly enriched peaks for  Acinar_1_2_6  cells relative to other celltypes.

Finished Acinar_1_2_6 2024-05-29 17:43:05

Starting Acinar_3 at 2024-05-29 17:43:05

Starting log reg Acinar_3 at 2024-05-29 17:43:28

Joining with `by = join_by(peak)`
There were  353593  significantly enriched peaks for  Acinar_3  cells relative to other celltypes.

Finished Acinar_3 2024-05-29 22:57:26

Starting Acinar_5 at 2024-05-29 22:57:26

Starting log reg Acinar_5 at 2024-05-29 22:57:51

Joining with `by = join_by(peak)`
There were  361808  significantly enriched peaks for  Acinar_5  cells relative to other celltypes.

Finished Acinar_5 2024-05-30 03:53:52

Starting Acinar_4 at 2024-05-30 03:53:52

Starting log reg Acinar_4 at 2024-05-30 03:54:16

Joining with `by = join_by(peak)`
There were  335466  significantly enriched peaks for  Acinar_4 

# Acinar subtpyes

## Log regression

In [27]:
#Read in meta data you want to use
metadata<-read.table('/nfs/lab/projects/nPOD/nPOD_metadata.txt', sep="\t", header=TRUE)
metadata$nPOD_ID <- as.character(metadata$Sample.ID)
metadata <- metadata[colnames(metadata) %in% c('nPOD_ID','Condition_subtype')]

In [28]:
#Define input and output directories
indir<-'pseudobulk_mtx/acinar_subtype/'
outdir<-'052824_correctedResults_allNDSamples_wAAb/acinar_subtypes'
dir.create(outdir)

In [29]:
#Function that basically transposes the data and merges it with metadata so
#that glm() can be used on a single dataframe.

#NOTE!! You will need to factorize the predicted variable and any covariates, in this case the Celltype and Sample
#(Donor), respectively.
get_input_df<-function(celltype, df_interest, df_other, meta_data){
    dfa<-as.data.frame(t(df_interest))
    dfo<-as.data.frame(t(df_other))
    dfa$nPOD_ID<-rownames(dfa)
    dfo$nPOD_ID<-rownames(dfo)
    dfa$Celltype<-celltype
    dfo$Celltype<-paste0("Not",celltype)
    dfall<-rbind(dfa,dfo)
    int<-left_join(x = dfall, y=metadata, by = 'nPOD_ID',multiple = "all")#,relationship = "many-to-many")
    int$Celltype<-as.factor(int$Celltype)
    int$nPOD_ID<-as.factor(int$nPOD_ID)
    final<-int
}

In [30]:
cells <- c('Acinar_1_2_6','Acinar_5','Acinar_3','Acinar_4')
cells

[1] "Acinar_1_2_6" "Acinar_5"     "Acinar_3"     "Acinar_4"

In [51]:
(res1)

peak,pvalue,std.error
<chr>,<chr>,<chr>
1:1000020-1000320,NA,NA
1:100028799-100029099,NA,NA
1:100037929-100038229,NA,NA
1:100038647-100038947,NA,NA
1:100046900-100047200,NA,NA
1:100051493-100051692,0.997361597746026,129123.250361249
1:100065849-100066097,0.997992425855333,188017.409341679
1:1000713-1001013,NA,NA
1:100071817-100072117,NA,NA


In [31]:
##Running logistic regression
meta <- read.table('/nfs/lab/projects/nPOD/nPOD_metadata.txt', sep='\t', header = TRUE)
all_disease <- meta[meta$Condition_subtype %in% c('T1D_early','T1D_late'),]$Sample.ID
samples <- unique(meta$Sample.ID)
nd <- setdiff(samples, all_disease)

for (cell in cells){
    message(paste0("Starting ",cell," at ", Sys.time()))
    #Read in inputs where "interest" is counts matrix for a given cell type and "other" is counts matrix for all
    #other cell types combined
    interest<-read.table(paste0(indir,cell,'_sample_gex_CMP_CORRECTED2.txt'), sep="\t", header=TRUE, check.names = FALSE)
    colnames(interest) <- gsub('X','',colnames(interest))
    nd_interest <- interest#[,colnames(interest)%in% nd]
    other<-read.table(paste0(indir,'all_but_',cell,'_sample_gex_CMP_CORRECTED2.txt'), sep="\t", header=TRUE,check.names = FALSE)
    colnames(other) <- gsub('X','',colnames(other))
    nd_other <- other#[,colnames(other)%in% nd]
    #Reads in files and gets input formatted
    input<-get_input_df(cell, nd_interest, nd_other, metadata)
    #Runs logistic regression in parallel on 20 cores (can change number of cores with mc.cores)
    res_list<-mclapply(1:(ncol(input)-3), function(i) {
        res<-tidy(glm(formula=Celltype ~ input[ ,i] + nPOD_ID, family="binomial", data=input))
        pvalue<-res[2,5]
        ster<-res[2,3]
        peak<-colnames(input)[i]
        c(peak,pvalue, ster)
    }, mc.cores=60)
    #Combines results list into dataframe
    res1<-matrix(unlist(res_list), ncol=3, byrow= TRUE)
    res1<-as.data.frame(res1)
    colnames(res1)<-c("peak","pvalue", "std.error")
    
    #Calculates log2foldchange from normalized counts matricies you read in and adds this into results dataframe
    L2FC_list<-mclapply(1:nrow(res1), function(i) {
        peak<-res1$peak[i]
        mean_interest<-mean(as.numeric(interest[peak,]))
        mean_other<-mean(as.numeric(other[peak, ]))
        FC<-(mean_interest-mean_other)/mean_other
        L2FC<-log2(abs(FC))
        c(peak,L2FC)
    }, mc.cores=60)
    addinL2FC<-matrix(unlist(L2FC_list), ncol=2, byrow=TRUE)
    addinL2FC<-as.data.frame(addinL2FC)
    colnames(addinL2FC)<-c('peak', 'L2FC')
    rownames(addinL2FC)<-addinL2FC$peak
    newdf<-left_join(res1, addinL2FC)
    newdf$pvalue<-as.numeric(newdf$pvalue)
    #Adds in bonferroni corrected padj values as a column
    numpeaks<-nrow(newdf)
    newdf$padj<-p.adjust(p = newdf$pvalue, method = "BH", n = nrow(newdf))
    sorteddf<-arrange(newdf, pvalue, desc(L2FC))
    #Writes the final table to output directory
    write.table(sorteddf, paste0(outdir,cell,'.LogRegOut.txt'),  sep="\t", quote=FALSE, row.names=FALSE)
    
    #Prints the number of significantly enriched peaks for each celltype based on padj ≤ 0.1
    numsig<-nrow(sorteddf[sorteddf$padj <= 0.1, ])
    message(paste("There were ", numsig, " significantly enriched peaks for ", cell, " cells relative to other celltypes."))
    message("Finished ", cell, " ", Sys.time())
}


Starting Acinar_1_2_6 at 2024-06-02 01:06:58

Joining with `by = join_by(peak)`
There were  337806  significantly enriched peaks for  Acinar_1_2_6  cells relative to other celltypes.

Finished Acinar_1_2_6 2024-06-02 06:11:45

Starting Acinar_5 at 2024-06-02 06:11:45

Joining with `by = join_by(peak)`
There were  355022  significantly enriched peaks for  Acinar_5  cells relative to other celltypes.

Finished Acinar_5 2024-06-02 11:40:35

Starting Acinar_3 at 2024-06-02 11:40:35

Joining with `by = join_by(peak)`
There were  309281  significantly enriched peaks for  Acinar_3  cells relative to other celltypes.

Finished Acinar_3 2024-06-02 17:00:04

Starting Acinar_4 at 2024-06-02 17:00:04

Joining with `by = join_by(peak)`
There were  329254  significantly enriched peaks for  Acinar_4  cells relative to other celltypes.

Finished Acinar_4 2024-06-02 21:56:05



In [33]:
outdir
getwd()

[1] "052824_correctedResults_allNDSamples_wAAb/acinar_subtypes"

[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724"